# Phase 3A: First and Second Moment Incremental Clustering (FMIC / SMIC)

US-3.19 and US-3.20 established independent FMIC and SMIC monitoring at alpha 0.10 on the authoritative 
stable cohort. US-3.22 evaluates a fresh full run for each approved alpha. Mathematical definitions are shared with the preserved 
Phase 2-scheduled clustering foundation; the Phase 2 baseline is comparison-only.


In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

## Stage 1: Input Loading and Policy Resolution

This stage validates input artifacts, resolves preferred and fallback source policies, and normalizes numeric columns required by the deterministic state machine.

Run context note: repository paths resolve from `Path.cwd()` first and fall back to its parent when the notebook is executed from the `scripts` directory. If artifacts still are not found under the resolved project root, run the notebook from the expected repository root or adjust path resolution.

In [ ]:
# Stage 1: Load the aligned input, experiment config, and authoritative Phase 3A threshold.
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / "artifacts").exists() and (candidate / "scripts").exists():
            return candidate
    return Path.cwd()

REPO_ROOT = resolve_repo_root()
PHASE3A_CONFIG_PATH = REPO_ROOT / "config/phase3a_experiment_config.json"
if not PHASE3A_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Required Phase 3A config not found: {PHASE3A_CONFIG_PATH}")
phase3a_experiment_config = json.loads(
    PHASE3A_CONFIG_PATH.read_text(encoding="utf-8")
)
PHASE1_ALIGNED_PATH = REPO_ROOT / phase3a_experiment_config["input_source"]
PHASE1_STATS_PATH = REPO_ROOT / phase3a_experiment_config["threshold"]["statistics_artifact"]
PHASE2_OUTPUT_PATH = REPO_ROOT / "artifacts/phase2_temperature_sensors.csv"
PHASE2_METRICS_PATH = REPO_ROOT / "artifacts/phase2_temperature_metrics.json"
PHASE3A_THRESHOLD_SERIES_NAME = phase3a_experiment_config["threshold"]["source_series"]
PRIMARY_SOURCE_SERIES_NAME = PHASE3A_THRESHOLD_SERIES_NAME
PRIMARY_GLOBAL_AVG_COLUMN = "average_temperature_all_devices"
SOURCE_SERIES_NAME = PRIMARY_SOURCE_SERIES_NAME
GLOBAL_AVG_COLUMN = PRIMARY_GLOBAL_AVG_COLUMN


def require_input_files(paths):
    for path in paths:
        if not path.exists():
            raise FileNotFoundError(f"Required input not found: {path}")


def resolve_global_average_column(df):
    if PRIMARY_GLOBAL_AVG_COLUMN in df.columns:
        return PRIMARY_GLOBAL_AVG_COLUMN
    raise ValueError(
        "Could not find the required all-devices global average column in aligned phase1 data"
    )


def select_p90_row(stats_frame):
    row = stats_frame.loc[stats_frame["series_name"] == PHASE3A_THRESHOLD_SERIES_NAME]
    if len(row) != 1:
        raise ValueError(
            "Expected exactly one phase3a_stable_cohort row in phase1_average_statistics.csv"
        )
    return row


def find_sensor_columns(df):
    cols = sorted(
        [
            column
            for column in df.columns
            if column.endswith("_temperature")
            and column != PRIMARY_GLOBAL_AVG_COLUMN
        ]
    )
    if not cols:
        raise ValueError("No per-sensor temperature columns found in aligned phase1 data")
    return cols


require_input_files([PHASE1_ALIGNED_PATH, PHASE1_STATS_PATH])
phase1_df = pd.read_csv(PHASE1_ALIGNED_PATH, keep_default_na=False, na_values=[""])
stats_df = pd.read_csv(PHASE1_STATS_PATH, float_precision="round_trip")

if phase1_df.empty:
    raise ValueError("aligned_phase1_temperature.csv has no rows")

GLOBAL_AVG_COLUMN = resolve_global_average_column(phase1_df)
p90_row = select_p90_row(stats_df)
p90_threshold = float(p90_row.iloc[0]["p90"])
configured_p90 = phase3a_experiment_config["threshold"].get("p90_value")
if configured_p90 is None or p90_threshold != float(configured_p90):
    raise ValueError(
        "Phase 3A config P90 does not match the authoritative statistics row."
    )
sensor_cols = find_sensor_columns(phase1_df)

import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from scripts.phase2_monitoring import normalize_phase2_readings
phase1_df = normalize_phase2_readings(phase1_df, [GLOBAL_AVG_COLUMN, *sensor_cols])

print(
    json.dumps(
        {
            "repo_root": str(REPO_ROOT),
            "rows": int(len(phase1_df)),
            "sensor_count": int(len(sensor_cols)),
            "phase3a_p90_threshold": p90_threshold,
            "threshold_series": SOURCE_SERIES_NAME,
            "threshold_status": phase3a_experiment_config["threshold"]["status"],
            "phase3a_config": str(PHASE3A_CONFIG_PATH),
            "phase1_input": str(PHASE1_ALIGNED_PATH),
            "global_avg_column": GLOBAL_AVG_COLUMN,
        },
        indent=2,
    )
)

## Stage 2: History, Normalization, Clustering, and Input Validation

All Phase 3 core definitions live in this notebook. The tagged cell below 
retains the validated mathematics and is also loaded by separate automated tests.


In [ ]:
"""Source-aligned history normalization and clustering for Phase 3A."""

from __future__ import annotations

from dataclasses import dataclass
import math
from numbers import Real
from types import MappingProxyType
from typing import Iterable, Mapping

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class HistoryNormalizationSnapshot:
    """One event-level normalization result shared by alpha candidates."""

    eligible_sensor_ids: tuple[str, ...]
    raw_histories: Mapping[str, float]
    history_magnitudes: Mapping[str, float]
    maximum_history_magnitude: float | None
    normalized_histories: Mapping[str, float] | None
    normalization_status: str
    partition_basis: str
    zero_history_fallback: bool
    fallback_reason: str | None
    fallback_partition: tuple[tuple[str, ...], ...] | None
    deferred_reason: str | None


@dataclass(frozen=True)
class AlphaClusteringInput:
    """Per-alpha handoff referencing the shared normalization snapshot."""

    clustering_event_id: int
    synchronization_id: int
    alpha: float
    normalization_snapshot: HistoryNormalizationSnapshot


PHASE3_ALPHA_VALUES = tuple(value / 100 for value in range(5, 100, 5))
PHASE3A_DEFAULT_ALPHA_VALUES = (0.10,)


def normalize_sensor_histories(
    raw_histories: Mapping[str, Real],
    eligible_sensor_ids: Iterable[str],
) -> HistoryNormalizationSnapshot:
    """Normalize eligible scalar histories without mutating persistent state."""
    if isinstance(eligible_sensor_ids, str):
        raise ValueError("Provide eligible sensor IDs as an iterable, not a string.")
    eligible_ids = tuple(eligible_sensor_ids)
    if any(not isinstance(sensor_id, str) or not sensor_id for sensor_id in eligible_ids):
        raise ValueError("Eligible sensor IDs must be non-empty strings.")
    if len(set(eligible_ids)) != len(eligible_ids):
        raise ValueError("Eligible sensor IDs must be unique.")

    if not eligible_ids:
        return HistoryNormalizationSnapshot(
            eligible_sensor_ids=(),
            raw_histories=MappingProxyType({}),
            history_magnitudes=MappingProxyType({}),
            maximum_history_magnitude=None,
            normalized_histories=None,
            normalization_status="deferred",
            partition_basis="not_applicable",
            zero_history_fallback=False,
            fallback_reason=None,
            fallback_partition=None,
            deferred_reason="no_eligible_sensors",
        )

    missing = sorted(set(eligible_ids) - set(raw_histories))
    if missing:
        raise ValueError(f"Raw history is missing eligible sensors: {missing}.")

    eligible_histories: dict[str, float] = {}
    for sensor_id in sorted(eligible_ids):
        value = raw_histories[sensor_id]
        if isinstance(value, bool) or not isinstance(value, Real):
            raise ValueError(f"History for {sensor_id} must be a finite number.")
        numeric_value = float(value)
        if not math.isfinite(numeric_value):
            raise ValueError(f"History for {sensor_id} must be finite.")
        eligible_histories[sensor_id] = numeric_value

    magnitudes = {
        sensor_id: abs(history)
        for sensor_id, history in eligible_histories.items()
    }
    maximum_magnitude = max(magnitudes.values())
    if maximum_magnitude == 0:
        return HistoryNormalizationSnapshot(
            eligible_sensor_ids=tuple(sorted(eligible_histories)),
            raw_histories=MappingProxyType(eligible_histories),
            history_magnitudes=MappingProxyType(magnitudes),
            maximum_history_magnitude=0.0,
            normalized_histories=None,
            normalization_status="all_zero_history_fallback",
            partition_basis="all_zero_history_fallback",
            zero_history_fallback=True,
            fallback_reason="all_eligible_histories_are_zero",
            fallback_partition=tuple(
                (sensor_id,) for sensor_id in sorted(eligible_histories)
            ),
            deferred_reason=None,
        )

    normalized = {
        sensor_id: history / maximum_magnitude
        for sensor_id, history in eligible_histories.items()
    }
    return HistoryNormalizationSnapshot(
        eligible_sensor_ids=tuple(sorted(eligible_histories)),
        raw_histories=MappingProxyType(eligible_histories),
        history_magnitudes=MappingProxyType(magnitudes),
        maximum_history_magnitude=maximum_magnitude,
        normalized_histories=MappingProxyType(normalized),
        normalization_status="normalized",
        partition_basis="normalized_history",
        zero_history_fallback=False,
        fallback_reason=None,
        fallback_partition=None,
        deferred_reason=None,
    )


def build_alpha_clustering_inputs(
    synchronization_id: int,
    normalization_snapshot: HistoryNormalizationSnapshot,
    alpha_values: Iterable[Real] = PHASE3A_DEFAULT_ALPHA_VALUES,
) -> tuple[AlphaClusteringInput, ...]:
    """Build selected alpha inputs, sharing one event normalization result."""
    if synchronization_id < 1:
        raise ValueError("Synchronization ID must be positive.")
    if normalization_snapshot.deferred_reason is not None:
        return ()
    return tuple(
        AlphaClusteringInput(
            clustering_event_id=synchronization_id,
            synchronization_id=synchronization_id,
            alpha=float(alpha),
            normalization_snapshot=normalization_snapshot,
        )
        for alpha in alpha_values
    )


Cluster = tuple[str, ...]
Partition = tuple[Cluster, ...]


def _validated_clustering_values(
    normalized_histories: Mapping[str, Real], alpha: Real,
) -> dict[str, float]:
    if isinstance(alpha, bool) or not isinstance(alpha, Real):
        raise ValueError("Alpha must be a finite number in [0, 1].")
    if not math.isfinite(float(alpha)) or not 0 <= alpha <= 1:
        raise ValueError("Alpha must be a finite number in [0, 1].")
    values: dict[str, float] = {}
    for sensor_id, value in normalized_histories.items():
        if not isinstance(sensor_id, str) or not sensor_id:
            raise ValueError("Clustering sensor IDs must be non-empty strings.")
        if isinstance(value, bool) or not isinstance(value, Real):
            raise ValueError(f"Normalized history for {sensor_id} must be finite.")
        numeric_value = float(value)
        if not math.isfinite(numeric_value) or abs(numeric_value) > 1:
            raise ValueError(f"Normalized history for {sensor_id} must be in [-1, 1].")
        values[sensor_id] = numeric_value
    return values


def _canonical_cluster(cluster: Iterable[str]) -> Cluster:
    if isinstance(cluster, str):
        raise ValueError("A cluster must be an iterable of sensor IDs, not a string.")
    members = tuple(cluster)
    if not members or any(not isinstance(member, str) or not member for member in members):
        raise ValueError("Clusters must contain non-empty string sensor IDs.")
    if len(set(members)) != len(members):
        raise ValueError("A cluster must not contain duplicate sensor IDs.")
    return tuple(sorted(members))


def _cluster_quality(cluster: Cluster, values: Mapping[str, float], alpha: float) -> float:
    return abs(math.fsum(values[sensor_id] for sensor_id in cluster)) / len(cluster) + alpha * len(cluster)


def cluster_quality(
    cluster: Iterable[str], normalized_histories: Mapping[str, Real], alpha: Real,
) -> float:
    """Score signed cancellation before applying the scalar absolute value."""
    values = _validated_clustering_values(normalized_histories, alpha)
    members = _canonical_cluster(cluster)
    unknown = sorted(set(members) - set(values))
    if unknown:
        raise ValueError(f"Cluster contains sensors without normalized histories: {unknown}.")
    return _cluster_quality(members, values, float(alpha))


def partition_quality(
    partition: Iterable[Iterable[str]], normalized_histories: Mapping[str, Real], alpha: Real,
) -> float:
    """Score a disjoint partition containing exactly the supplied sensors."""
    values = _validated_clustering_values(normalized_histories, alpha)
    clusters = tuple(sorted(_canonical_cluster(cluster) for cluster in partition))
    members = tuple(member for cluster in clusters for member in cluster)
    if not clusters or len(set(members)) != len(members) or set(members) != set(values):
        raise ValueError("Partition must cover every eligible sensor exactly once.")
    return max(_cluster_quality(cluster, values, float(alpha)) for cluster in clusters)


def evaluate_alpha_clustering(clustering_input: AlphaClusteringInput) -> dict[str, object] | None:
    """Retain the greedy candidate sequence and choose its earliest minimum."""
    snapshot = clustering_input.normalization_snapshot
    if snapshot.deferred_reason is not None:
        return None
    eligible = snapshot.eligible_sensor_ids
    if not eligible:
        raise ValueError("A completed clustering input must have eligible sensors.")
    if snapshot.zero_history_fallback:
        if any(snapshot.raw_histories[sensor_id] != 0 for sensor_id in eligible):
            raise ValueError("All-zero fallback requires zero raw eligible histories.")
        # Zero raw histories allow fallback scoring without fabricating normalization.
        source_values = snapshot.raw_histories
    else:
        if snapshot.normalized_histories is None:
            raise ValueError("Nonzero history clustering requires normalized histories.")
        source_values = snapshot.normalized_histories
    values = _validated_clustering_values(source_values, clustering_input.alpha)
    if len(set(eligible)) != len(eligible) or set(values) != set(eligible):
        raise ValueError("Clustering values must cover exactly the eligible sensor set.")
    alpha = float(clustering_input.alpha)
    partition: Partition = tuple((sensor_id,) for sensor_id in sorted(eligible))
    candidates: list[dict[str, object]] = []
    candidate_qualities: list[float] = []
    candidate_partitions: list[Partition] = []
    merges: list[dict[str, object]] = []

    while True:
        qualities = {cluster: _cluster_quality(cluster, values, alpha) for cluster in partition}
        quality = max(qualities.values())
        candidates.append({
            "candidate_index": len(candidates),
            "partition": partition,
            "cluster_count": len(partition),
            "cluster_qualities": [
                {"members": cluster, "quality": qualities[cluster]} for cluster in partition
            ],
            "partition_quality": quality,
        })
        candidate_qualities.append(quality)
        candidate_partitions.append(partition)
        if snapshot.zero_history_fallback or len(partition) <= 2:
            break

        worst = min(partition, key=lambda cluster: (-qualities[cluster], cluster))
        partners = []
        for partner in partition:
            if partner != worst:
                merged = tuple(sorted(worst + partner))
                partners.append((partner, merged, _cluster_quality(merged, values, alpha)))
        partner, merged, merged_quality = min(partners, key=lambda item: (item[2], item[0]))
        merges.append({
            "from_candidate_index": len(candidates) - 1,
            "worst_cluster": worst,
            "worst_cluster_quality": qualities[worst],
            "partner_scores": [
                {"partner": item[0], "merged_cluster": item[1], "merged_quality": item[2]}
                for item in partners
            ],
            "selected_partner": partner,
            "merged_cluster": merged,
            "merged_quality": merged_quality,
        })
        partition = tuple(sorted(
            tuple(cluster for cluster in partition if cluster not in (worst, partner)) + (merged,)
        ))

    selected_index = min(range(len(candidates)), key=lambda index: candidate_qualities[index])
    selected_partition = candidate_partitions[selected_index]
    return {
        "clustering_event_id": clustering_input.clustering_event_id,
        "synchronization_id": clustering_input.synchronization_id,
        "normalization_snapshot_id": clustering_input.synchronization_id,
        "alpha": alpha,
        "partition_basis": snapshot.partition_basis,
        "zero_history_fallback": snapshot.zero_history_fallback,
        "fallback_reason": snapshot.fallback_reason,
        "candidate_partitions": candidates,
        "merge_steps": merges,
        "selected_candidate_index": selected_index,
        "selected_partition": selected_partition,
        "partition_quality": candidate_qualities[selected_index],
        "cluster_count": len(selected_partition),
        "cluster_assignments": {
            sensor_id: cluster_index
            for cluster_index, cluster in enumerate(selected_partition)
            for sensor_id in cluster
        },
    }

PHASE3A_APPROVED_SENSOR_IDS = ("A", "C", "D", "E", "F", "G")
PHASE3A_BUCKET_SECONDS = 60


def validate_phase3a_config(config):
    """Reject configuration drift from the approved stable-cohort contract."""
    if not isinstance(config, dict):
        raise ValueError("Phase 3A experiment configuration must be a JSON object.")
    if config.get("input_source") != "artifacts/aligned_phase1_temperature.csv":
        raise ValueError("Phase 3A input must use the aligned Phase 1 artifact.")

    sensor_ids = tuple(config.get("selected_sensor_ids", ()))
    if sensor_ids != PHASE3A_APPROVED_SENSOR_IDS:
        raise ValueError(
            "Phase 3A cohort must remain exactly A, C, D, E, F, and G."
        )
    expected_columns = [f"{sensor_id}_temperature" for sensor_id in sensor_ids]
    if config.get("selected_sensor_temperature_columns") != expected_columns:
        raise ValueError(
            "Phase 3A temperature columns must match the approved sensor cohort."
        )
    if config.get("endpoint_convention") != "inclusive":
        raise ValueError("Phase 3A interval endpoints must both be inclusive.")
    if config.get("sampling_interval_seconds") != PHASE3A_BUCKET_SECONDS:
        raise ValueError("Phase 3A sampling interval must be exactly 60 seconds.")
    if config.get("stable_cohort_required") is not True:
        raise ValueError("Phase 3A must require the fixed stable cohort.")
    if config.get("missing_value_policy") != "fail":
        raise ValueError("Phase 3A missing observations must fail validation.")
    if config.get("imputation_policy") != "none":
        raise ValueError("Phase 3A input must not use imputed sensor values.")
    if config.get("baseline_alpha") != 0.10:
        raise ValueError("The Phase 3A baseline alpha must be 0.10.")
    if config.get("boundary_convention") != {
        "safe_when": "metric < threshold",
        "violated_when": "metric >= threshold",
    }:
        raise ValueError("Phase 3A boundary convention does not match the approved policy.")

    threshold_config = config.get("threshold")
    if not isinstance(threshold_config, dict):
        raise ValueError("Phase 3A threshold configuration must be an object.")
    if threshold_config.get("source_series") != "phase3a_stable_cohort":
        raise ValueError("The Phase 3A threshold source must be phase3a_stable_cohort.")
    if threshold_config.get("statistic") != "p90":
        raise ValueError("The authoritative Phase 3A threshold statistic must be p90.")
    threshold_value = threshold_config.get("p90_value")
    if isinstance(threshold_value, bool) or not isinstance(
        threshold_value, (int, float)
    ) or not np.isfinite(threshold_value):
        raise ValueError("The authoritative Phase 3A P90 must be a finite number.")
    if threshold_config.get("status") != "authoritative":
        raise ValueError("The Phase 3A P90 must be marked authoritative.")
    if threshold_config.get("statistics_artifact") != "artifacts/phase1_average_statistics.csv":
        raise ValueError("Phase 3A P90 provenance must name the Phase 1 statistics artifact.")
    if threshold_config.get("production_script") != "scripts/build_phase1_temperature.py":
        raise ValueError("Phase 3A P90 provenance must name the Phase 1 production script.")

    start_time = pd.Timestamp(config.get("start_timestamp_utc"))
    end_time = pd.Timestamp(config.get("end_timestamp_utc"))
    if start_time.tz is None or end_time.tz is None:
        raise ValueError("Phase 3A interval timestamps must be timezone-aware.")
    start_time = start_time.tz_convert("UTC")
    end_time = end_time.tz_convert("UTC")
    if end_time <= start_time:
        raise ValueError("Phase 3A end timestamp must follow its start timestamp.")

    elapsed_seconds = (end_time - start_time).total_seconds()
    if elapsed_seconds % PHASE3A_BUCKET_SECONDS:
        raise ValueError("Phase 3A endpoints must align to one-minute buckets.")
    expected_elapsed_minutes = int(elapsed_seconds // 60)
    expected_bucket_count = config.get("expected_bucket_count")
    if isinstance(expected_bucket_count, bool) or not isinstance(
        expected_bucket_count, int
    ):
        raise ValueError("Phase 3A expected_bucket_count must be an integer.")
    if expected_bucket_count != expected_elapsed_minutes + 1:
        raise ValueError("Phase 3A bucket count must include both interval endpoints.")
    if config.get("expected_elapsed_minutes") != expected_elapsed_minutes:
        raise ValueError("Phase 3A expected elapsed minutes do not match its endpoints.")

    return sensor_ids, expected_columns, start_time, end_time, expected_bucket_count


def prepare_phase3a_input(source_frame, experiment_config):
    """Select and validate the approved complete one-minute Phase 3A input."""
    sensor_ids, sensor_columns, start_time, end_time, expected_bucket_count = (
        validate_phase3a_config(experiment_config)
    )
    required_columns = ["bucket_epoch", "bucket_time_utc", *sensor_columns]
    missing_columns = sorted(set(required_columns) - set(source_frame.columns))
    if missing_columns:
        raise ValueError(f"Phase 3A input is missing required columns: {missing_columns}")

    timestamps = pd.to_datetime(
        source_frame["bucket_time_utc"], utc=True, errors="coerce"
    )
    if timestamps.isna().any():
        raise ValueError("Phase 1 input contains an invalid bucket_time_utc timestamp.")
    in_interval = timestamps.between(start_time, end_time, inclusive="both")
    selected = source_frame.loc[in_interval, required_columns].copy()
    selected["bucket_time_utc"] = pd.to_datetime(
        selected["bucket_time_utc"], utc=True, errors="raise"
    )

    if len(selected) != expected_bucket_count:
        raise ValueError(
            f"Phase 3A interval requires {expected_bucket_count} buckets; "
            f"found {len(selected)}."
        )
    selected_times = selected["bucket_time_utc"]
    if not selected_times.is_unique:
        raise ValueError("Phase 3A input contains duplicate timestamps.")
    if not selected_times.is_monotonic_increasing:
        raise ValueError("Phase 3A timestamps must be in strictly increasing order.")
    if selected_times.iloc[0] != start_time:
        raise ValueError("Phase 3A first timestamp does not match the approved start.")
    if selected_times.iloc[-1] != end_time:
        raise ValueError("Phase 3A final timestamp does not match the approved end.")

    time_deltas = selected_times.diff().dt.total_seconds().iloc[1:]
    if not time_deltas.eq(PHASE3A_BUCKET_SECONDS).all():
        bad_position = int(
            np.flatnonzero(
                ~time_deltas.eq(PHASE3A_BUCKET_SECONDS).to_numpy()
            )[0]
        ) + 1
        raise ValueError(
            f"Phase 3A timestamp gap at selected row {bad_position}; "
            "expected exactly 60 seconds."
        )

    epochs = pd.to_numeric(selected["bucket_epoch"], errors="coerce")
    epoch_values = epochs.to_numpy(dtype=float)
    if not np.isfinite(epoch_values).all() or not np.equal(
        epoch_values, np.floor(epoch_values)
    ).all():
        raise ValueError("Phase 3A bucket_epoch values must be finite whole seconds.")
    expected_epochs = np.fromiter(
        (int(timestamp.timestamp()) for timestamp in selected_times),
        dtype=np.int64,
        count=len(selected_times),
    )
    if not np.array_equal(epochs.to_numpy(dtype=np.int64), expected_epochs):
        raise ValueError("Phase 3A bucket_epoch values do not match timestamps.")
    if not epochs.diff().iloc[1:].eq(PHASE3A_BUCKET_SECONDS).all():
        raise ValueError("Phase 3A bucket_epoch values must be consecutive 60-second steps.")
    selected["bucket_epoch"] = epochs.astype("int64")

    for column in sensor_columns:
        numeric = pd.to_numeric(selected[column], errors="coerce")
        invalid = numeric.isna() | ~np.isfinite(numeric.to_numpy(dtype=float))
        if invalid.any():
            row_number = int(np.flatnonzero(invalid.to_numpy())[0])
            raise ValueError(
                f"Phase 3A requires a finite {column} value in every bucket; "
                f"missing or invalid value at {selected_times.iloc[row_number]}."
            )
        selected[column] = numeric.astype(float)
    return selected.reset_index(drop=True)


class Phase3AHistoryTracker:
    """Track signed accumulated history for an always-present fixed cohort."""

    DISCOUNT = 0.5

    def __init__(self, sensor_ids):
        self.sensor_ids = tuple(sensor_ids)
        if not self.sensor_ids or len(set(self.sensor_ids)) != len(self.sensor_ids):
            raise ValueError("Phase 3A sensor IDs must be non-empty and unique.")
        self.histories = {sensor_id: 0.0 for sensor_id in self.sensor_ids}
        self.references = {sensor_id: None for sensor_id in self.sensor_ids}
        self.synchronization_id = 0
        self.row_number = 0

    def step(self, readings, synchronization_causes=()):
        if set(readings) != set(self.sensor_ids):
            raise ValueError("Phase 3A readings must contain exactly the fixed cohort.")
        values = {}
        for sensor_id in self.sensor_ids:
            value = readings[sensor_id]
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.number)):
                raise ValueError(f"Phase 3A reading for {sensor_id} must be finite and numeric.")
            value = float(value)
            if not np.isfinite(value):
                raise ValueError(f"Phase 3A reading for {sensor_id} must be finite and numeric.")
            values[sensor_id] = value

        if isinstance(synchronization_causes, str):
            raise ValueError("Synchronization causes must be an iterable, not a string.")
        causes = tuple(sorted(set(synchronization_causes)))
        if self.synchronization_id == 0 and not causes:
            raise ValueError("The first Phase 3A row must complete initialization synchronization.")
        if self.synchronization_id:
            for sensor_id in self.sensor_ids:
                self.histories[sensor_id] = (
                    self.DISCOUNT * self.histories[sensor_id]
                    + values[sensor_id] - self.references[sensor_id]
                )
        clustering_histories = None
        if causes:
            self.synchronization_id += 1
            clustering_histories = dict(self.histories)
            self.references = dict(values)
        self.row_number += 1
        return clustering_histories, causes



## Stage 3: Shared FMIC / SMIC Allocation and Monitoring

Source-defined first/second moments, the labeled project zero-weight fallback, and the shared node/coordinator/root engine. Import-only tests load this tagged cell after the mathematical definitions; no production run occurs here.

### First-moment provenance and recurrence

Source: [Clustering for Monitoring Distributed Data Streams, section 1.4, printed page 10 / PDF page 14](../Writing/Motivation/140516editor%20Book%20Chapter%20%28Last%20Version%29.pdf#page=14), the discounted vector set immediately before the moment-based node constraints. For observation index `m >= 1`, the source set is `V'_n(m) = {2^(-(m-i)) * v_n(t_i) : i=1,...,m}` and `mu_n` is its arithmetic mean. Section 1.5.2 identifies FMIC as using only the first moment.

Therefore `S_n(m) = sum(i=1..m, 2^(-(m-i)) * v_n(t_i)) = 0.5*S_n(m-1) + v_n(t_m)` and `mu_n(m) = S_n(m)/m`, with `S_n(0)=0`. The recurrence is an algebraic implementation of that source set, not a separately quoted source equation. Divide by the observation count `m`, not by the sum of discount weights; this is not an exponentially weighted average.

Project sampling convention: initialization `t0` installs references but is not a moment sample. Every subsequent one-minute raw reading contributes, including synchronization rows. Neither `S_n` nor `m` resets at synchronization, and `m` is not a synchronization count. Before the first sample, the implementation exposes zero moments to the separately labeled project allocation fallback.

Hand calculation for post-initialization readings A=`[2,4,6]`, B=`[-4,2,3]`:

| m | Discounted A observations | S_A | mu_A | Discounted B observations | S_B | mu_B |
|---|---|---|---|---|---|---|
| 1 | [2] | 2 | 2 | [-4] | -4 | -4 |
| 2 | [1,4] | 5 | 5/2 | [-2,2] | 0 | 0 |
| 3 | [0.5,2,6] | 8.5 | 8.5/3 | [-1,1,3] | 3 | 1 |

The separate clustering history is `h_n(t)=0.5*h_n(t-1)+(v_n(t)-reference_n)`, not a raw-reading moment and not divided by `m`. FMIC applies `abs(mu_n)` only when computing weights; signed first moments are preserved. The hand-calculated regression is `FirstMomentTests.test_discounted_arithmetic_mean_is_not_history_or_ema` in [the monitoring tests](test_phase3_monitoring.py); synchronization persistence is checked separately by `test_history_and_first_moment_persist_across_sync`.

### SMIC second-moment provenance and stable recurrence

The **rendered** source section 1.4 (printed p.10/PDF 14) defines `sigma_n^2` as a sum of squared deviations without a divisor, but prints an inconsistent index `i=0..m`. [Appendix 1, printed p.22/PDF 26](../Writing/Motivation/140516editor%20Book%20Chapter%20%28Last%20Version%29.pdf#page=26) resolves this: `sigma_n^2 = sum(i=1..m, (v'_n(t_i)-mu_n)^2)`. We use those `m` observations, with **no division by m or m-1**. Despite the source's term 'central second moment', this quantity is not conventional population/sample variance.

[Appendix 1, printed p.24/PDF 28](../Writing/Motivation/140516editor%20Book%20Chapter%20%28Last%20Version%29.pdf#page=28) gives a stable update: with `k` previous observations, old mean `mu`, and new raw value `x`, `sigma_new^2 = 0.25*sigma_old^2 + k/(k+1)*(x-0.5*mu)^2`. It scales every previous vector by 1/2 and appends x. This reuses the unchanged first-moment update, keeps one common observation count, and never resets at root synchronization. Both terms are nonnegative, so no numerical cancellation correction or negative-value clamp is used; nonfinite or negative states fail explicitly.

Hand example A from above: at m=1, sigma^2=0 and W=2; at m=2, squared deviations `[9/4,9/4]` sum to `9/2`, giving `W=5/2+sqrt((1/2)*(9/2))=4`; at m=3, `[49/9,25/36,361/36]` sum to `97/6`, giving `W=17/6+sqrt(97/9)`. `SecondMomentTests.test_source_discounted_squared_deviations_and_weights_by_hand` verifies both sensors at every step.

SMIC uses `W_n=abs(mu_n)+sqrt(((m-1)/m)*sigma_n^2)`, then the same normalized node and mean coordinator allowances as FMIC. At m=0, zero moments and equal weights are an explicit project initialization convention; at m=1 the central sum and conservative contribution are zero. The approved project all-zero fallback applies only if the total weight is zero; an individual zero weight otherwise stays zero.

`FMICMonitor` retains its compatibility name but is the single lifecycle for `weight_method='FMIC'` or `'SMIC'`. Moment/weight payloads accompany existing sensor-to-root synchronization responses and allowances accompany existing bundled updates. No extra exchanges or internal-computation messages are charged. References, clustering, signed cancellation, escalation, next-bucket activation, and two-sided correctness policy are shared. Each method generates its own schedule and writes distinct compact outputs; FMIC schemas remain unchanged.


In [ ]:
"""Shared FMIC/SMIC monitoring for the fixed Phase 3A experiment.

Chapter section 1.4, printed p.10 (PDF p.14): mu is the arithmetic
mean of the discounted vectors {v(t_m), v(t_(m-1))/2, ...}.
FMIC keeps only |mu| in the general moment weight (section 1.5.2).
Clustering history and the first moment are deliberately separate states.
"""

from __future__ import annotations

import hashlib
import json
import math
import tempfile
from collections.abc import Callable, Mapping, Sequence
from numbers import Real
from pathlib import Path

import pandas as pd


MESSAGE_CATEGORIES = (
    "node_trigger_messages",
    "cluster_check_requests",
    "cluster_check_responses",
    "coordinator_root_escalations",
    "root_sync_requests",
    "root_sync_responses",
    "root_state_updates",
)


def fmic_allocation(
    moments: Mapping[str, float],
    partition: Sequence[Sequence[str]],
    root_allowance: float,
) -> dict:
    if not moments or any(
        isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value)
        for value in moments.values()
    ):
        raise ValueError("FMIC moments must be nonempty and finite.")
    weights = {sensor: abs(value) for sensor, value in moments.items()}
    return {
        "first_moments": dict(moments),
        **allocate_moment_weights(weights, partition, root_allowance, "source_first_moment"),
    }


def allocate_moment_weights(
    weights: Mapping[str, float],
    partition: Sequence[Sequence[str]],
    root_allowance: float,
    weight_status: str,
) -> dict:
    if not weights or any(not math.isfinite(value) or value < 0 for value in weights.values()):
        raise ValueError("Raw moment weights must be nonempty, finite, and nonnegative.")
    if not math.isfinite(root_allowance) or root_allowance < 0:
        raise ValueError("Root allowance must be finite and nonnegative.")
    members = [sensor for cluster in partition for sensor in cluster]
    if any(not cluster for cluster in partition) or sorted(members) != sorted(weights):
        raise ValueError("Partition must cover every sensor exactly once.")
    total = math.fsum(weights.values())
    if not math.isfinite(total):
        raise ValueError("Total moment weight overflowed.")
    fallback = total == 0
    scaled = {
        sensor: 1.0 if fallback else len(weights) * (value / total)
        for sensor, value in weights.items()
    }
    node = {sensor: value * root_allowance for sensor, value in scaled.items()}
    if any(not math.isfinite(value) for value in node.values()):
        raise ValueError("Moment node allowances overflowed.")
    coordinator = {
        max(cluster): math.fsum(node[sensor] for sensor in cluster) / len(cluster)
        for cluster in partition
    }
    return {
        "raw_weights": dict(weights),
        "scaled_weights": scaled,
        "node_allowances": node,
        "coordinator_allowances": coordinator,
        "weight_status": "project_all_zero_equal_weight_fallback" if fallback else weight_status,
    }


class DiscountedFirstMoment:
    """Implement the source discounted vector-set mean (section 1.4, p.10/PDF 14).

    S(m) = sum(2**(-(m-i)) * raw_reading(i), i=1..m), mu(m) = S(m)/m.
    Updating S by 0.5*S + reading is algebraically equivalent to that set.
    Count every post-t0 observation, not sync events; neither state resets.
    This is not a weight-normalized EMA or the reference-relative history.
    See the preceding provenance cell and FirstMomentTests' hand calculation.
    """

    def __init__(self, sensors: Sequence[str]):
        self.sums = dict.fromkeys(sensors, 0.0)
        self.count = 0

    def update(self, readings: Mapping[str, float]) -> None:
        if set(readings) != set(self.sums):
            raise ValueError("First moments require the fixed sensor cohort.")
        if any(
            isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value)
            for value in readings.values()
        ):
            raise ValueError("First-moment observations must be finite.")
        self.count += 1
        self.sums = {
            sensor: 0.5 * self.sums[sensor] + readings[sensor]
            for sensor in self.sums
        }
        if any(not math.isfinite(value) for value in self.sums.values()):
            raise ValueError("Discounted first-moment sums overflowed.")

    def means(self) -> dict[str, float]:
        return {
            sensor: value / self.count if self.count else 0.0
            for sensor, value in self.sums.items()
        }


class DiscountedSecondMoment(DiscountedFirstMoment):
    """Source sum of squared deviations, NOT population/sample variance.

    Appendix 1, printed p.22/PDF 26: sigma^2 = sum((x_i-mu)^2), i=1..m.
    This resolves section 1.4's i=0 indexing typo; there is no divisor.
    Appendix 1, printed p.24/PDF 28, supplies the stable recurrence below.
    Scaling prior vectors by 1/2 scales their squared deviations by 1/4.
    All update terms are nonnegative: no cancellation clamp is needed.
    """

    def __init__(self, sensors: Sequence[str]):
        super().__init__(sensors)
        self.second_moments = dict.fromkeys(self.sums, 0.0)

    def update(self, readings: Mapping[str, float]) -> None:
        old_means = self.means()
        old_count = self.count
        super().update(readings)
        updated = {}
        for sensor, prior in self.second_moments.items():
            delta = readings[sensor] - 0.5 * old_means[sensor]
            updated[sensor] = 0.25 * prior + (old_count / self.count) * delta * delta
        if any(not math.isfinite(value) or value < 0 for value in updated.values()):
            raise ValueError("Discounted second moments must remain finite and nonnegative.")
        self.second_moments = updated


def smic_allocation(
    moments: Mapping[str, float],
    second_moments: Mapping[str, float],
    count: int,
    partition: Sequence[Sequence[str]],
    root_allowance: float,
) -> dict:
    if isinstance(count, bool) or not isinstance(count, int) or count < 0:
        raise ValueError("SMIC observation count must be a nonnegative integer.")
    if not moments or set(moments) != set(second_moments):
        raise ValueError("SMIC first and second moments must cover the same fixed cohort.")
    if any(
        isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(value)
        for value in (*moments.values(), *second_moments.values())
    ) or any(value < 0 for value in second_moments.values()):
        raise ValueError("SMIC moments must be finite; second moments must be nonnegative.")
    if count == 0 and any(value != 0 for value in moments.values()):
        raise ValueError("No observations must give zero first moments.")
    if count <= 1 and any(value != 0 for value in second_moments.values()):
        raise ValueError("Zero or one observation must give zero central second moments.")
    factor = (count - 1) / count if count else 0.0
    terms = {sensor: math.sqrt(factor * value) for sensor, value in second_moments.items()}
    weights = {sensor: abs(value) + terms[sensor] for sensor, value in moments.items()}
    return {
        "first_moments": dict(moments),
        "second_moments": dict(second_moments),
        "conservative_terms": terms,
        **allocate_moment_weights(weights, partition, root_allowance, "source_first_plus_second_moment"),
    }


class FMICMonitor:
    """Shared FMIC/SMIC lifecycle; compatibility name, no imported sync schedule.

    allocation_strategy(moment, partition, root_allowance) optionally injects
    allocation values for protocol tests; otherwise weight_method selects them.
    """

    def __init__(
        self,
        sensor_ids: Sequence[str],
        threshold: float,
        alpha: float = 0.10,
        *,
        clustering: Callable = evaluate_alpha_clustering,
        weight_method: str = "FMIC",
        allocation_strategy: Callable | None = None,
        raise_on_false_safe: bool = True,
        capture_decisions: bool = False,
    ):
        if weight_method not in ("FMIC", "SMIC"):
            raise ValueError("weight_method must be FMIC or SMIC.")
        self.weight_method = weight_method
        self.allocation_strategy = allocation_strategy
        self.raise_on_false_safe = raise_on_false_safe
        if not isinstance(capture_decisions, bool):
            raise ValueError("capture_decisions must be a boolean.")
        self.capture_decisions = capture_decisions
        self.sensors = tuple(sensor_ids)
        if not self.sensors or len(set(self.sensors)) != len(self.sensors):
            raise ValueError("Sensors must be nonempty and unique.")
        if not math.isfinite(threshold) or not math.isfinite(alpha) or not 0 <= alpha <= 1:
            raise ValueError("Threshold and alpha must be finite; alpha must be in [0, 1].")
        self.threshold = threshold
        self.alpha = alpha
        self.clustering = clustering
        self.history = Phase3AHistoryTracker(self.sensors)
        self.moment = (
            DiscountedSecondMoment(self.sensors) if weight_method == "SMIC"
            else DiscountedFirstMoment(self.sensors)
        )
        self.references: dict[str, float] = {}
        self.partition: tuple[tuple[str, ...], ...] = ()
        self.allocation: dict = {}
        self.known_violated = False
        self.events: list[dict] = []
        self.decision_trace: list[dict] = []
        self.messages = dict.fromkeys(MESSAGE_CATEGORIES, 0)
        self.rows = 0
        self.node_triggers = 0
        self.checks = 0
        self.escalations = 0
        self.truth_violations = 0
        self.false_safe = 0
        self.false_violated = 0

    def step(self, readings: Mapping[str, float], timestamp: str, epoch: int) -> dict:
        if set(readings) != set(self.sensors):
            raise ValueError(f"{self.weight_method} readings must contain exactly the fixed cohort.")
        if any(isinstance(value, bool) or not isinstance(value, Real) for value in readings.values()):
            raise ValueError(f"{self.weight_method} observations must be numeric, not booleans or text.")
        values = {sensor: float(readings[sensor]) for sensor in self.sensors}
        if any(not math.isfinite(value) for value in values.values()):
            raise ValueError(f"{self.weight_method} observations must be finite.")
        counts = dict.fromkeys(MESSAGE_CATEGORIES, 0)
        entry_references = dict(self.references)
        entry_node_allowances = dict(self.allocation.get("node_allowances", {}))
        entry_coordinator_allowances = dict(self.allocation.get("coordinator_allowances", {}))
        entry_partition = self.partition
        entry_known_state = (
            None if not self.references else "VIOLATED" if self.known_violated else "SAFE"
        )
        initialization = not self.references
        triggering_sensor_ids = set()
        check_count = escalation_count = trigger_count = 0
        if not initialization:
            self.moment.update(values)
            for cluster in self.partition:
                coordinator = max(cluster)
                deviations = {
                    sensor: values[sensor] - self.references[sensor] for sensor in cluster
                }
                triggers = {
                    sensor for sensor in cluster
                    if abs(deviations[sensor]) >= self.allocation["node_allowances"][sensor]
                }
                if not triggers:
                    continue
                triggering_sensor_ids.update(triggers)
                trigger_count += len(triggers)
                check_count += 1
                # All triggering non-coordinators send their current reading.
                # The coordinator requests only peers not already reporting.
                counts["node_trigger_messages"] += len(triggers - {coordinator})
                requested = set(cluster) - triggers - {coordinator}
                counts["cluster_check_requests"] += len(requested)
                counts["cluster_check_responses"] += len(requested)
                cluster_change = abs(math.fsum(deviations.values()) / len(cluster))
                if cluster_change >= self.allocation["coordinator_allowances"][coordinator]:
                    escalation_count += 1
                    counts["coordinator_root_escalations"] += 1
        synchronize = initialization or escalation_count > 0
        causes = ("initialization",) if initialization else (
            ("coordinator_violation",) if synchronize else ()
        )
        histories, _ = self.history.step(values, synchronization_causes=causes)
        actual_mean = math.fsum(values.values()) / len(values)
        actual_violated = actual_mean >= self.threshold
        if synchronize:
            # The separate root has no sensor readings. Alerts carry no raw
            # readings; one collection obtains every sensor's state directly.
            counts["root_sync_requests"] = len(self.sensors)
            counts["root_sync_responses"] = len(self.sensors)
            counts["root_state_updates"] = len(self.sensors)
            if histories is None:
                raise RuntimeError("Synchronization did not produce a history snapshot.")
            snapshot = normalize_sensor_histories(histories, self.sensors)
            evaluation = self.clustering(
                AlphaClusteringInput(
                    self.history.synchronization_id,
                    self.history.synchronization_id, self.alpha, snapshot,
                )
            )
            if evaluation is None:
                raise RuntimeError(f"{self.weight_method} synchronization unexpectedly deferred clustering.")
            self.partition = tuple(tuple(cluster) for cluster in evaluation["selected_partition"])
            root_allowance = abs(self.threshold - actual_mean)
            if self.allocation_strategy is not None:
                self.allocation = self.allocation_strategy(self.moment, self.partition, root_allowance)
            elif isinstance(self.moment, DiscountedSecondMoment):
                self.allocation = smic_allocation(
                    self.moment.means(), self.moment.second_moments, self.moment.count,
                    self.partition, root_allowance,
                )
            else:
                self.allocation = fmic_allocation(self.moment.means(), self.partition, root_allowance)
            self.references = dict(values)
            self.known_violated = actual_violated
            event = {
                "clustering_event_id": self.history.synchronization_id,
                "synchronization_id": self.history.synchronization_id,
                "bucket_epoch": epoch, "bucket_time_utc": timestamp,
                "algorithm": self.weight_method, "alpha": self.alpha,
                "synchronization_causes": "|".join(causes),
                "raw_sensor_histories": json.dumps(histories, sort_keys=True),
                "normalized_histories": json.dumps(
                    dict(snapshot.normalized_histories)
                    if snapshot.normalized_histories is not None else None, sort_keys=True,
                ),
                "normalization_status": snapshot.normalization_status,
                "selected_partition": json.dumps(self.partition),
                "selected_partition_quality": evaluation["partition_quality"],
                "cluster_count": len(self.partition),
                "coordinator_assignments": json.dumps(
                    {str(i): max(cluster) for i, cluster in enumerate(self.partition)}
                ),
                "coordinator_rule": "lexicographically_highest_member",
                "global_average": actual_mean, "global_violated": int(actual_violated),
                "p90_threshold": self.threshold, "root_allowance": root_allowance,
                "moment_observation_count": self.moment.count,
                "weight_status": self.allocation["weight_status"],
            }
            for field in (
                "first_moments", "raw_weights", "scaled_weights",
                "node_allowances", "coordinator_allowances",
            ):
                event[field] = json.dumps(self.allocation[field], sort_keys=True)
            if self.weight_method == "SMIC":
                for field in ("second_moments", "conservative_terms"):
                    event[field] = json.dumps(self.allocation[field], sort_keys=True)
            self.events.append(event)
        false_safe = actual_violated and not self.known_violated
        self.rows += 1
        self.truth_violations += int(actual_violated)
        self.false_safe += int(false_safe)
        self.false_violated += int(not actual_violated and self.known_violated)
        self.node_triggers += trigger_count
        self.checks += check_count
        self.escalations += escalation_count
        for category in counts:
            self.messages[category] += counts[category]
        if self.capture_decisions:
            self.decision_trace.append({
                "bucket_epoch": epoch,
                "bucket_time_utc": timestamp,
                "entry_references": entry_references,
                "entry_node_allowances": entry_node_allowances,
                "entry_coordinator_allowances": entry_coordinator_allowances,
                "entry_partition": entry_partition,
                "entry_known_state": entry_known_state,
                "triggering_sensor_ids": tuple(sorted(triggering_sensor_ids)),
                "synchronized": synchronize,
                "observed_global_average": actual_mean,
                "global_violated": actual_violated,
                "root_known_state": "VIOLATED" if self.known_violated else "SAFE",
                "post_references": dict(self.references),
                "post_partition": self.partition,
                "post_node_allowances": dict(self.allocation.get("node_allowances", {})),
                "post_coordinator_allowances": dict(
                    self.allocation.get("coordinator_allowances", {})
                ),
                "messages": dict(counts),
                "total_messages": sum(counts.values()),
            })
        if false_safe and self.raise_on_false_safe:
            raise RuntimeError(f"Blocking {self.weight_method} false-safe at {timestamp}.")
        return {
            "synchronized": synchronize, "global_violated": actual_violated,
            "known_violated": self.known_violated, "false_safe": false_safe,
            "node_triggers": trigger_count, "coordinator_checks": check_count,
            "coordinator_escalations": escalation_count, "messages": counts,
            "total_messages": sum(counts.values()),
        }

    def summary(self) -> dict:
        if not self.rows:
            raise ValueError(f"Cannot summarize an uninitialized {self.weight_method} monitor.")
        cluster_sizes = [
            len(cluster) for event in self.events
            for cluster in json.loads(event["selected_partition"])
        ]
        return {
            "algorithm": self.weight_method, "alpha": self.alpha,
            "bucket_count": self.rows,
            "total_messages": sum(self.messages.values()),
            "message_category_totals": dict(self.messages),
            "root_synchronization_count": len(self.events),
            "initialization_count": 1,
            "monitoring_triggered_synchronization_count": len(self.events) - 1,
            "node_trigger_count": self.node_triggers,
            "coordinator_check_count": self.checks,
            "coordinator_to_root_escalation_count": self.escalations,
            "global_threshold_violation_count": self.truth_violations,
            "false_safe_count": self.false_safe,
            "false_violated_count": self.false_violated,
            "correctness_status": "passed" if self.false_safe == 0 and self.false_violated == 0 else "failed",
            "message_counts_reconcile": True,
            "cluster_count_min": min(event["cluster_count"] for event in self.events),
            "cluster_count_max": max(event["cluster_count"] for event in self.events),
            "cluster_size_min": min(cluster_sizes),
            "cluster_size_max": max(cluster_sizes),
            "cluster_size_mean_across_sync_clusters": math.fsum(cluster_sizes) / len(cluster_sizes),
            "root_model": "separate_root_component",
            "monitoring_policy": "absolute distance to threshold; root-known state retained until sync",
            "moment_policy": "all post-initialization raw readings; discounted_sum = 0.5 * prior_sum + reading; mu = discounted_sum / sample_count; never reset at sync",
            "weight_zero_policy": "project fallback: all W_n=0 gives w_n=1; individual zero weights remain zero",
            "protocol": "all clusters checked once per bucket; all triggering peers report; collect remaining peers; alerts carry no raw readings; one root collection per bucket; updates take effect next bucket",
            "correctness_policy": "centralized truth is validation-only and never triggers communication",
            "source": "Clustering for Monitoring Distributed Data Streams, section 1.4 p.10-11, section 1.5.2; separate root and >= boundary are project conventions",
        }


def run_phase3a_fmic(
    root: Path,
    alpha: float | None = None,
    output_dir: Path | None = None,
    *,
    prepared_inputs: dict | None = None,
    raise_on_false_safe: bool = True,
) -> dict:
    return run_phase3a_monitoring(
        root, "FMIC", alpha, output_dir, prepared_inputs=prepared_inputs,
        raise_on_false_safe=raise_on_false_safe,
    )


def run_phase3a_smic(
    root: Path,
    alpha: float | None = None,
    output_dir: Path | None = None,
    *,
    prepared_inputs: dict | None = None,
    raise_on_false_safe: bool = True,
) -> dict:
    return run_phase3a_monitoring(
        root, "SMIC", alpha, output_dir, prepared_inputs=prepared_inputs,
        raise_on_false_safe=raise_on_false_safe,
    )


def load_phase3a_monitoring_inputs(root: Path) -> dict:
    config_path = root / "config" / "phase3a_experiment_config.json"
    config = json.loads(config_path.read_text(encoding="utf-8"))
    sensors, _, _, _, _ = validate_phase3a_config(config)
    threshold_config = config["threshold"]
    threshold = float(threshold_config["p90_value"])
    statistics_path = root / threshold_config["statistics_artifact"]
    statistics = pd.read_csv(statistics_path, float_precision="round_trip")
    threshold_rows = statistics.loc[
        statistics["series_name"] == threshold_config["source_series"], "p90"
    ]
    if len(threshold_rows) != 1 or float(threshold_rows.iloc[0]) != threshold:
        raise ValueError("Phase 3A threshold does not match the authoritative statistics artifact.")
    input_path = root / config["input_source"]
    selected = prepare_phase3a_input(
        pd.read_csv(input_path, float_precision="round_trip"), config
    )
    baseline_path = root / "artifacts" / "phase3a_phase2_baseline.json"
    baseline = json.loads(baseline_path.read_text(encoding="utf-8"))
    for key in (
        "experiment_name", "input_source", "selected_sensor_ids", "start_timestamp_utc",
        "end_timestamp_utc", "endpoint_convention", "boundary_convention",
    ):
        if baseline[key] != config[key]:
            raise ValueError(f"Phase 2 baseline does not match {key}.")
    if (
        baseline["bucket_count"] != len(selected)
        or baseline["threshold_value"] != threshold
        or baseline["threshold_source_series"] != threshold_config["source_series"]
        or baseline["correctness_status"] != "passed"
        or baseline["algorithm"] != "phase2"
        or sum(baseline["message_category_totals"].values()) != baseline["total_messages"]
    ):
        raise ValueError("Invalid matched Phase 2 baseline for Phase 3A comparison.")
    return {
        "root": root.resolve(), "config": config, "sensors": sensors,
        "threshold": threshold, "threshold_config": threshold_config,
        "selected": selected, "baseline": baseline,
        "config_path": config_path, "input_path": input_path,
        "statistics_path": statistics_path,
        "baseline_path": baseline_path,
    }


def run_phase3a_monitoring(
    root: Path,
    weight_method: str,
    alpha: float | None = None,
    output_dir: Path | None = None,
    *,
    prepared_inputs: dict | None = None,
    raise_on_false_safe: bool = True,
) -> dict:
    if weight_method not in ("FMIC", "SMIC"):
        raise ValueError("weight_method must be FMIC or SMIC.")
    inputs = prepared_inputs or load_phase3a_monitoring_inputs(root)
    if inputs["root"] != root.resolve():
        raise ValueError("Prepared Phase 3A inputs belong to a different repository root.")
    config = inputs["config"]
    sensors = inputs["sensors"]
    threshold = inputs["threshold"]
    threshold_config = inputs["threshold_config"]
    selected = inputs["selected"]
    baseline = inputs["baseline"]
    alpha = float(config["baseline_alpha"] if alpha is None else alpha)
    if not math.isfinite(alpha) or not 0 <= alpha <= 1:
        raise ValueError("Production alpha must be finite and in [0, 1].")
    if output_dir is None and alpha != float(config["baseline_alpha"]):
        raise ValueError("A non-baseline alpha requires an explicit output directory to preserve baseline artifacts.")

    monitor = FMICMonitor(
        sensors, threshold, alpha, weight_method=weight_method,
        raise_on_false_safe=raise_on_false_safe,
    )
    for row in selected.to_dict("records"):
        monitor.step(
            {sensor: row[f"{sensor}_temperature"] for sensor in sensors},
            row["bucket_time_utc"].isoformat(), int(row["bucket_epoch"]),
        )
    summary = monitor.summary()
    if summary["global_threshold_violation_count"] != baseline["global_violation_count"]:
        raise ValueError(f"{weight_method} centralized truth does not match the Phase 2 baseline.")
    summary.update({
        key: config[key] for key in (
            "experiment_name", "input_source", "selected_sensor_ids",
            "start_timestamp_utc", "end_timestamp_utc", "endpoint_convention",
            "boundary_convention",
        )
    })
    summary.update({
        "threshold_source_series": threshold_config["source_series"],
        "threshold_value": threshold,
        "threshold_statistic": threshold_config["statistic"],
        "threshold_statistics_artifact": threshold_config["statistics_artifact"],
        "phase2_total_messages": baseline["total_messages"],
        "phase2_resynchronizations": baseline["resynchronization_count"],
        "centralized_sensor_transmission_reference": len(selected) * len(sensors),
        "message_reduction_vs_phase2_percent": (
            100 * (1 - summary["total_messages"] / baseline["total_messages"])
        ),
        "message_reduction_vs_centralized_percent": (
            100 * (1 - summary["total_messages"] / (len(selected) * len(sensors)))
        ),
        "input_sha256": hashlib.sha256(inputs["input_path"].read_bytes()).hexdigest(),
        "config_sha256": hashlib.sha256(inputs["config_path"].read_bytes()).hexdigest(),
        "phase2_baseline_sha256": hashlib.sha256(inputs["baseline_path"].read_bytes()).hexdigest(),
    })
    if weight_method == "SMIC":
        summary.update({
            "second_moment_policy": "source unnormalized sum of squared deviations; no m or m-1 divisor; persistent; 0.25*prior + k/(k+1)*(reading - 0.5*old_mu)^2",
            "second_moment_source": "Appendix 1 printed p.22/PDF 26 defines i=1..m, resolving section 1.4's i=0 typo; recurrence printed p.24/PDF 28",
            "second_moment_numerics": "nonnegative stable update; no negative-variance clamp or tolerance",
            "moment_payload_policy": "sensor moments/weights piggyback on existing root-sync responses; allowances on existing state updates; no additional transmissions",
            "comparison_status": "fixed-alpha comparison with matched Phase 2; not a universal algorithm ranking",
        })
    target_dir = output_dir or (root / "artifacts")
    target_dir.mkdir(parents=True, exist_ok=True)
    events_path = target_dir / f"phase3a_{weight_method.lower()}_events.csv"
    summary_path = target_dir / f"phase3a_{weight_method.lower()}_summary.json"
    events = pd.DataFrame.from_records(monitor.events)
    events["threshold_source_series"] = threshold_config["source_series"]
    events.to_csv(events_path, index=False)
    summary["events_sha256"] = hashlib.sha256(events_path.read_bytes()).hexdigest()
    summary_path.write_text(json.dumps(summary, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return summary


def _phase3a_cluster_metrics(events: pd.DataFrame, sensor_count: int) -> dict:
    if events.empty:
        raise ValueError("A monitoring run must include its initialization clustering event.")
    cluster_counts = [len(json.loads(value)) for value in events["selected_partition"]]
    cluster_sizes = [
        len(cluster)
        for partition_value in events["selected_partition"]
        for cluster in json.loads(partition_value)
    ]
    memberships = sum(cluster_sizes)
    if memberships != len(events) * sensor_count:
        raise ValueError("Synchronization partitions must cover each sensor exactly once.")
    return {
        "cluster_count_min": min(cluster_counts),
        "cluster_count_max": max(cluster_counts),
        "cluster_count_mean_per_sync_event": math.fsum(cluster_counts) / len(events),
        "cluster_size_min": min(cluster_sizes),
        "cluster_size_max": max(cluster_sizes),
        "cluster_size_mean_across_sync_clusters": memberships / len(cluster_sizes),
    }


def _phase3a_validate_us321(root: Path) -> dict:
    path = root / "artifacts" / "phase3a_validation_summary.json"
    if not path.exists():
        raise FileNotFoundError(f"US-3.21 validation record not found: {path}")
    validation = json.loads(path.read_text(encoding="utf-8"))
    runs = validation.get("production_replays", [])
    if (
        validation.get("story") != "US-3.21"
        or validation.get("test_result", {}).get("status") != "passed"
        or len(runs) != 2
        or any(run.get("correctness_status") != "passed" for run in runs)
        or any(run.get("message_counts_reconcile") is not True for run in runs)
    ):
        raise ValueError("US-3.21 validation gate is not recorded as passed for both algorithms.")
    return validation


def _phase3a_alpha_plots(results_path: Path, output_dir: Path, phase2_messages: int) -> list[str]:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    results = pd.read_csv(results_path)
    if len(results) != 38 or results[["algorithm", "alpha_label"]].duplicated().any():
        raise ValueError("Saved alpha results must contain 38 unique algorithm/alpha rows.")
    passing = results.loc[results["correctness_status"] == "passed"]
    plots = (
        ("total_messages", "Actual transmissions", "phase3a_messages_vs_alpha.png"),
        ("root_synchronization_count", "Root synchronizations (including initialization)", "phase3a_synchronizations_vs_alpha.png"),
        ("cluster_count_mean_per_sync_event", "Mean clusters per synchronization event", "phase3a_clustering_vs_alpha.png"),
    )
    for metric, ylabel, filename in plots:
        figure, axis = plt.subplots(figsize=(8, 5), constrained_layout=True)
        for algorithm, color in (("FMIC", "#1f77b4"), ("SMIC", "#d62728")):
            subset = passing.loc[passing["algorithm"] == algorithm].sort_values("alpha")
            axis.plot(subset["alpha"], subset[metric], marker="o", color=color, label=f"{algorithm} (correctness passed)")
        if metric == "total_messages":
            axis.axhline(phase2_messages, color="#444444", linestyle="--", label="Matched Phase 2 (fixed benchmark)")
        axis.set_xlabel("Alpha (tested grid: 0.05 to 0.95 by 0.05)")
        axis.set_ylabel(ylabel)
        axis.set_title(f"Phase 3A fixed A/C/D/E/F/G cohort: {ylabel} vs alpha")
        axis.set_xticks([step / 20 for step in range(1, 20)])
        axis.grid(True, alpha=0.25)
        axis.legend()
        figure.savefig(output_dir / filename, dpi=160)
        plt.close(figure)
    return [filename for _, _, filename in plots]


def run_phase3a_alpha_sweep(root: Path) -> dict:
    validation = _phase3a_validate_us321(root)
    inputs = load_phase3a_monitoring_inputs(root)
    alpha_steps = tuple(range(1, 20))
    alpha_grid = tuple(step / 20 for step in alpha_steps)
    alpha_labels = tuple(f"{step / 20:.2f}" for step in alpha_steps)
    if len(set(alpha_grid)) != 19 or alpha_labels[1] != "0.10":
        raise RuntimeError("The approved alpha grid is not unique or lacks the baseline alpha.")

    artifacts_dir = root / "artifacts"
    notebook_path = root / "scripts" / "build_phase3_temperature_clustering.ipynb"
    if not notebook_path.exists():
        notebook_path = Path(globals().get("NOTEBOOK_PATH", Path.cwd() / "scripts" / "build_phase3_temperature_clustering.ipynb"))
    if not notebook_path.is_file():
        raise FileNotFoundError(f"Phase 3A notebook not found for provenance: {notebook_path}")
    protected_paths = [
        inputs["input_path"], inputs["statistics_path"], inputs["config_path"], inputs["baseline_path"],
        artifacts_dir / "phase3a_fmic_events.csv",
        artifacts_dir / "phase3a_fmic_summary.json",
        artifacts_dir / "phase3a_smic_events.csv",
        artifacts_dir / "phase3a_smic_summary.json",
    ]
    protected_hashes = {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in protected_paths
    }
    baseline_summaries = {
        method: json.loads((artifacts_dir / f"phase3a_{method.lower()}_summary.json").read_text(encoding="utf-8"))
        for method in ("FMIC", "SMIC")
    }
    output_rows = []
    event_frames = []
    category_names = tuple(MESSAGE_CATEGORIES)
    with tempfile.TemporaryDirectory(prefix="phase3a-alpha-sweep-") as temporary:
        temporary_dir = Path(temporary)
        for method, runner in (("FMIC", run_phase3a_fmic), ("SMIC", run_phase3a_smic)):
            for step, alpha, alpha_label in zip(alpha_steps, alpha_grid, alpha_labels):
                summary = runner(
                    root, alpha, temporary_dir, prepared_inputs=inputs,
                    raise_on_false_safe=False,
                )
                events_path = temporary_dir / f"phase3a_{method.lower()}_events.csv"
                events = pd.read_csv(events_path, float_precision="round_trip")
                if (
                    len(events) != summary["root_synchronization_count"]
                    or summary["total_messages"] != sum(summary["message_category_totals"].values())
                    or summary["message_counts_reconcile"] is not True
                ):
                    raise ValueError(f"{method} alpha {alpha_label} event or message totals do not reconcile.")
                cluster_metrics = _phase3a_cluster_metrics(
                    events, len(inputs["sensors"])
                )
                if alpha_label == "0.10":
                    expected = baseline_summaries[method]
                    for key in (
                        "total_messages", "message_category_totals",
                        "root_synchronization_count", "false_safe_count",
                        "false_violated_count", "correctness_status",
                    ):
                        if summary[key] != expected[key]:
                            raise RuntimeError(f"{method} alpha=0.10 regression in {key}.")
                    if summary["events_sha256"] != expected["events_sha256"]:
                        raise RuntimeError(f"{method} alpha=0.10 event fingerprint changed.")
                row = {
                    "algorithm": method,
                    "alpha": alpha,
                    "alpha_label": alpha_label,
                    "selected_sensor_ids": json.dumps(inputs["sensors"], separators=(",", ":")),
                    "start_timestamp_utc": inputs["config"]["start_timestamp_utc"],
                    "end_timestamp_utc": inputs["config"]["end_timestamp_utc"],
                    "endpoint_convention": inputs["config"]["endpoint_convention"],
                    "boundary_convention": json.dumps(inputs["config"]["boundary_convention"], sort_keys=True, separators=(",", ":")),
                    "bucket_count": summary["bucket_count"],
                    "threshold_value": summary["threshold_value"],
                    "correctness_status": summary["correctness_status"],
                    "false_safe_count": summary["false_safe_count"],
                    "false_violated_count": summary["false_violated_count"],
                    "global_threshold_violation_count": summary["global_threshold_violation_count"],
                    "total_messages": summary["total_messages"],
                    "initialization_count": summary["initialization_count"],
                    "monitoring_triggered_synchronization_count": summary["monitoring_triggered_synchronization_count"],
                    "root_synchronization_count": summary["root_synchronization_count"],
                    "node_trigger_count": summary["node_trigger_count"],
                    "coordinator_check_count": summary["coordinator_check_count"],
                    "coordinator_to_root_escalation_count": summary["coordinator_to_root_escalation_count"],
                    "message_counts_reconcile": summary["message_counts_reconcile"],
                    "phase2_total_messages": inputs["baseline"]["total_messages"],
                    "reduction_vs_phase2_percent": summary["message_reduction_vs_phase2_percent"],
                    "centralized_sensor_transmission_reference": summary["centralized_sensor_transmission_reference"],
                    "reduction_vs_centralized_percent": summary["message_reduction_vs_centralized_percent"],
                    **cluster_metrics,
                    "input_sha256": summary["input_sha256"],
                    "config_sha256": summary["config_sha256"],
                    "phase2_baseline_sha256": summary["phase2_baseline_sha256"],
                    "events_sha256": summary["events_sha256"],
                    "notebook_sha256": hashlib.sha256(notebook_path.read_bytes()).hexdigest(),
                    "failure_reason": "" if summary["correctness_status"] == "passed" else "Correctness gate failed.",
                }
                row.update(summary["message_category_totals"])
                output_rows.append(row)
                events["alpha_label"] = alpha_label
                event_frames.append(events)

    results = pd.DataFrame.from_records(output_rows).sort_values(
        ["algorithm", "alpha"], kind="stable"
    ).reset_index(drop=True)
    expected_pairs = {(method, label) for method in ("FMIC", "SMIC") for label in alpha_labels}
    actual_pairs = set(zip(results["algorithm"], results["alpha_label"]))
    if len(results) != 38 or actual_pairs != expected_pairs or results.duplicated(["algorithm", "alpha_label"]).any():
        raise ValueError("Alpha sweep did not produce exactly 38 unique approved algorithm/alpha results.")
    if any(results[category].lt(0).any() for category in category_names):
        raise ValueError("Message category totals cannot be negative.")
    results["reconciled_total_messages"] = results[list(category_names)].sum(axis=1)
    if not results["reconciled_total_messages"].eq(results["total_messages"]).all():
        raise ValueError("At least one alpha result does not reconcile to transmission categories.")

    output_dir = artifacts_dir
    output_dir.mkdir(parents=True, exist_ok=True)
    results_path = output_dir / "phase3a_alpha_results.csv"
    events_path = output_dir / "phase3a_alpha_events.csv"
    results.to_csv(results_path, index=False, float_format="%.17g")
    pd.concat(event_frames, ignore_index=True).to_csv(events_path, index=False)
    plots = _phase3a_alpha_plots(
        results_path, output_dir, inputs["baseline"]["total_messages"]
    )

    minima = {}
    for method in ("FMIC", "SMIC"):
        eligible = results.loc[
            (results["algorithm"] == method) & (results["correctness_status"] == "passed")
        ]
        if eligible.empty:
            minima[method] = {"status": "no correctness-passing alpha", "alphas": []}
        else:
            minimum = int(eligible["total_messages"].min())
            tied = eligible.loc[eligible["total_messages"] == minimum]
            minima[method] = {
                "status": "minimum observed on this dataset and tested grid",
                "minimum_total_messages": minimum,
                "alphas": tied["alpha_label"].tolist(),
                "representative_alpha_if_needed": tied["alpha_label"].min(),
                "results": tied.to_dict("records"),
            }
    paired = []
    for label in alpha_labels:
        fmic = results.loc[(results["algorithm"] == "FMIC") & (results["alpha_label"] == label)].iloc[0]
        smic = results.loc[(results["algorithm"] == "SMIC") & (results["alpha_label"] == label)].iloc[0]
        paired.append({
            "alpha": label,
            "fmic_minus_smic_total_messages": int(fmic["total_messages"] - smic["total_messages"]),
            "fmic_minus_smic_root_synchronizations": int(fmic["root_synchronization_count"] - smic["root_synchronization_count"]),
            "fmic_minus_smic_node_triggers": int(fmic["node_trigger_count"] - smic["node_trigger_count"]),
            "fmic_minus_smic_coordinator_checks": int(fmic["coordinator_check_count"] - smic["coordinator_check_count"]),
            "message_category_differences_fmic_minus_smic": {
                category: int(fmic[category] - smic[category]) for category in category_names
            },
        })
    trends = {}
    for method in ("FMIC", "SMIC"):
        values = results.loc[results["algorithm"] == method].sort_values("alpha")["total_messages"].tolist()
        if all(left <= right for left, right in zip(values, values[1:])):
            trend = "monotonic nondecreasing"
        elif all(left >= right for left, right in zip(values, values[1:])):
            trend = "monotonic nonincreasing"
        else:
            trend = "irregular across the tested grid"
        trends[method] = trend
    protected_after = {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in protected_paths
    }
    if protected_after != protected_hashes:
        raise RuntimeError("The alpha sweep modified a protected baseline or input artifact.")
    summary = {
        "story": "US-3.22",
        "experiment_name": inputs["config"]["experiment_name"],
        "alpha_grid": list(alpha_labels),
        "run_count": len(results),
        "cohort": inputs["sensors"],
        "start_timestamp_utc": inputs["config"]["start_timestamp_utc"],
        "end_timestamp_utc": inputs["config"]["end_timestamp_utc"],
        "endpoint_convention": inputs["config"]["endpoint_convention"],
        "bucket_count": len(inputs["selected"]),
        "threshold_source_series": inputs["threshold_config"]["source_series"],
        "threshold_value": inputs["threshold"],
        "boundary_convention": inputs["config"]["boundary_convention"],
        "phase2_baseline": inputs["baseline"],
        "centralized_sensor_transmission_reference": len(inputs["selected"]) * len(inputs["sensors"]),
        "baseline_alpha_results": results.loc[results["alpha_label"] == "0.10"].to_dict("records"),
        "minimum_observed_by_algorithm": minima,
        "paired_fmic_smic_differences": paired,
        "message_sensitivity_trend": trends,
        "cluster_size_aggregation": "Include the initialization partition; mean cluster size is total sensor memberships across all selected partitions at synchronization events divided by total clusters across those events.",
        "mean_cluster_count_aggregation": "Arithmetic mean of cluster counts, one count per synchronization event including initialization.",
        "correctness_status_by_algorithm": {
            method: "passed" if results.loc[results["algorithm"] == method, "correctness_status"].eq("passed").all() else "failed"
            for method in ("FMIC", "SMIC")
        },
        "message_accounting_status": "passed" if results["message_counts_reconcile"].all() else "failed",
        "us321_validation": {
            "status": validation["test_result"]["status"],
            "tests": validation["test_result"]["tests"],
        },
        "provenance": {
            "input_sha256": output_rows[0]["input_sha256"],
            "config_sha256": output_rows[0]["config_sha256"],
            "phase2_baseline_sha256": output_rows[0]["phase2_baseline_sha256"],
            "notebook_sha256": output_rows[0]["notebook_sha256"],
            "protected_artifact_sha256_before_and_after": protected_hashes,
        },
        "outputs": {
            "results_csv": results_path.name,
            "events_csv": events_path.name,
            "plots": plots,
        },
        "limitations": [
            "One deterministic six-sensor temperature dataset and one threshold; results do not imply statistical significance or generalization.",
            "Minimum observed alpha is an in-sample descriptive result on this dataset and tested grid, not a universal or theoretically optimal setting.",
            "No intermittent sensor availability or Phase 3B behavior was evaluated.",
        ],
    }
    summary_path = output_dir / "phase3a_comparison_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return summary


## Stage 4: Validation, Execution, and Results

The legacy Phase 2-scheduled helper remains for regression coverage only. Normal execution validates the US-3.21 prerequisite and runs the US-3.22 alpha sweep through the shared engine. Fixed alpha-0.10 event and summary artifacts are regression references and are not rewritten. Each method generates its own synchronization schedule.


In [ ]:
# Stage 5: Validate the stable experiment input before monitoring or clustering.
if "build_phase2_trace" not in globals():
    from scripts.phase2_monitoring import build_phase2_trace



def build_phase3a_history_events(
    source_frame,
    threshold,
    *,
    experiment_config,
    alpha_values=None,
    phase2_builder=build_phase2_trace,
):
    """Build compact event rows using the validated experiment configuration."""
    sensor_ids, sensor_columns, _, _, _ = validate_phase3a_config(experiment_config)
    authoritative_threshold = float(experiment_config["threshold"]["p90_value"])
    if float(threshold) != authoritative_threshold:
        raise ValueError("Supplied Phase 3A threshold does not match the authoritative config P90.")
    selected = prepare_phase3a_input(source_frame, experiment_config)
    if alpha_values is None:
        alpha_values = (float(experiment_config["baseline_alpha"]),)
    else:
        alpha_values = tuple(alpha_values)
    if not alpha_values:
        raise ValueError("At least one alpha value is required.")

    selected_times = selected["bucket_time_utc"]
    validation_summary = {
        "row_count": len(selected),
        "first_timestamp": selected_times.iloc[0].isoformat(),
        "last_timestamp": selected_times.iloc[-1].isoformat(),
        "selected_sensor_ids": list(sensor_ids),
        "missing_value_count": int(selected[sensor_columns].isna().sum().sum()),
        "timestamp_gap_count": int(
            selected_times.diff().dt.total_seconds().iloc[1:].ne(
                experiment_config["sampling_interval_seconds"]
            ).sum()
        ),
    }
    print("Phase 3A input validation: " + json.dumps(validation_summary, sort_keys=True))

    # The existing Phase 2 monitor supplies synchronization timing only; this is
    # not a separate Phase 2 comparison run or output artifact.
    baseline = phase2_builder(selected, sensor_columns, threshold)
    if len(baseline) != len(selected):
        raise ValueError("Phase 2 synchronization trace does not match Phase 3A input rows.")
    if not baseline["global_state_evaluable"].astype(bool).all():
        raise ValueError("Phase 3A fixed-cohort input unexpectedly has an undefined global state.")
    if baseline["observed_online_sensor_count"].ne(len(sensor_ids)).any():
        raise ValueError("Phase 3A input violated the fixed six-sensor cohort assumption.")
    if baseline["event_membership_changed"].astype(bool).any():
        raise ValueError("Phase 3A input unexpectedly changed sensor membership.")

    tracker = Phase3AHistoryTracker(sensor_ids)
    event_rows = []
    for row_index, input_row in selected.iterrows():
        baseline_row = baseline.iloc[row_index]
        if int(input_row["bucket_epoch"]) != int(baseline_row["bucket_epoch"]):
            raise ValueError(f"Phase 2/Phase 3A bucket mismatch at row {row_index}.")
        causes = ()
        if int(baseline_row["event_resync_performed"]) == 1:
            causes = tuple(
                cause for cause in str(baseline_row["event_resync_causes"]).split("|") if cause
            )
        histories, causes = tracker.step(
            {
                sensor_id: input_row[f"{sensor_id}_temperature"]
                for sensor_id in sensor_ids
            },
            synchronization_causes=causes,
        )
        if (histories is not None) != (int(baseline_row["event_resync_performed"]) == 1):
            raise ValueError(f"Phase 3A history synchronization mismatch at row {row_index}.")
        if histories is None:
            continue

        normalization = normalize_sensor_histories(histories, sensor_ids)
        alpha_inputs = build_alpha_clustering_inputs(
            tracker.synchronization_id, normalization, alpha_values
        )
        for alpha_input in alpha_inputs:
            evaluation = evaluate_alpha_clustering(alpha_input)
            if evaluation is None:
                raise RuntimeError("A completed stable-cohort event was unexpectedly deferred.")
            partition = evaluation["selected_partition"]
            coordinators = {
                str(cluster_index): max(cluster)
                for cluster_index, cluster in enumerate(partition)
            }
            event_rows.append({
                "clustering_event_id": tracker.synchronization_id,
                "synchronization_id": tracker.synchronization_id,
                "bucket_epoch": int(input_row["bucket_epoch"]),
                "bucket_time_utc": input_row["bucket_time_utc"].isoformat(),
                "synchronization_causes": "|".join(causes),
                "alpha": alpha_input.alpha,
                "raw_sensor_histories": json.dumps(dict(normalization.raw_histories), sort_keys=True),
                "normalized_histories": json.dumps(
                    dict(normalization.normalized_histories)
                    if normalization.normalized_histories is not None else None,
                    sort_keys=True,
                ),
                "normalization_status": normalization.normalization_status,
                "partition_basis": normalization.partition_basis,
                "zero_history_fallback": int(normalization.zero_history_fallback),
                "selected_partition": json.dumps(partition),
                "selected_partition_quality": evaluation["partition_quality"],
                "cluster_count": evaluation["cluster_count"],
                "coordinator_assignments": json.dumps(coordinators, sort_keys=True),
                "coordinator_rule": "lexicographically_highest_member",
                "p90_threshold": authoritative_threshold,
                "threshold_source_series": experiment_config["threshold"]["source_series"],
                "threshold_status": experiment_config["threshold"]["status"],
            })
    return pd.DataFrame.from_records(event_rows), baseline


def build_phase3a_phase2_baseline_summary(trace, experiment_config):
    """Summarize and validate the matched Phase 2 trace for Phase 3A."""
    expected_count = experiment_config["expected_bucket_count"]
    if len(trace) != expected_count:
        raise ValueError("Matched Phase 2 trace has the wrong Phase 3A bucket count.")

    sensor_count = len(experiment_config["selected_sensor_ids"])
    observed_counts = trace["observed_online_sensor_count"]
    if not observed_counts.eq(sensor_count).all():
        raise ValueError("Matched Phase 2 sensor count changed during the stable run.")
    membership_changes = int(trace["event_membership_changed"].sum())
    membership_syncs = int(trace["event_resync_triggered_by_membership_change"].sum())
    if membership_changes or membership_syncs:
        raise ValueError("Matched Phase 2 run unexpectedly reported membership changes.")

    category_columns = (
        "trigger_message_count",
        "request_message_count",
        "response_message_count",
        "broadcast_message_count",
    )
    category_totals = {
        column: int(trace[column].sum()) for column in category_columns
    }
    total_messages = int(trace["total_message_count"].sum())
    if sum(category_totals.values()) != total_messages:
        raise ValueError("Matched Phase 2 communication categories do not reconcile.")

    global_evaluation_count = int(trace["global_state_evaluable"].sum())
    if global_evaluation_count != expected_count:
        raise ValueError("Matched Phase 2 did not evaluate every Phase 3A bucket.")

    threshold = experiment_config["threshold"]
    return {
        "experiment_name": experiment_config["experiment_name"],
        "input_source": experiment_config["input_source"],
        "algorithm": "phase2",
        "selected_sensor_ids": experiment_config["selected_sensor_ids"],
        "start_timestamp_utc": experiment_config["start_timestamp_utc"],
        "end_timestamp_utc": experiment_config["end_timestamp_utc"],
        "endpoint_convention": experiment_config["endpoint_convention"],
        "boundary_convention": experiment_config["boundary_convention"],
        "bucket_count": expected_count,
        "global_evaluation_count": global_evaluation_count,
        "threshold_source_series": threshold["source_series"],
        "threshold_statistic": threshold["statistic"],
        "threshold_value": float(threshold["p90_value"]),
        "total_messages": total_messages,
        "message_category_totals": category_totals,
        "resynchronization_count": int(trace["event_resync_performed"].sum()),
        "membership_change_event_count": membership_changes,
        "membership_triggered_synchronization_count": membership_syncs,
        "observed_sensor_count_min": int(observed_counts.min()),
        "observed_sensor_count_max": int(observed_counts.max()),
        "global_violation_count": int(trace["global_violation"].sum()),
        "centralized_sensor_transmission_reference": expected_count * sensor_count,
        "message_counts_reconcile": True,
        "correctness_status": "passed",
    }


validate_phase3a_config(phase3a_experiment_config)
if globals().get("RUN_PHASE3A_EXECUTION", True):
    phase3a_alpha_summary = run_phase3a_alpha_sweep(REPO_ROOT)
    print(json.dumps(phase3a_alpha_summary, indent=2))
